# CS 3892 / 5892 — HW3 · Program analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/cs3892-examples/blob/main/notebooks/hw3-program-analysis.ipynb)

**Out Mon Oct 26 · due Tue Nov 3, 11:59 p.m.** The handout is
[`assignments/hw3.md`](https://github.com/ttj/cs3892-fmstai-fall2026/blob/main/assignments/hw3.md) in the class
repository. This notebook runs every starter as it ships, with the placeholders in place. **Edit
the files** (in Colab: the folder icon, left) **or the cells, and re-run.** Nothing here is an answer.

## Setup

Run these two once.

In [ ]:
# --- Setup: find the repo (clone on Colab), install Z3, fetch CBMC, define helpers
import os, subprocess, sys, pathlib

REPO_URL = "https://github.com/ttj/cs3892-examples.git"
HW = "hw3-program-analysis"

def _find_repo():
    here = pathlib.Path.cwd()
    for p in [here, *here.parents]:
        if (p / "homework" / HW).is_dir():
            return p
    dest = pathlib.Path("/content/cs3892-examples") if pathlib.Path("/content").is_dir() \
           else pathlib.Path.cwd() / "cs3892-examples"
    if not (dest / "homework" / HW).is_dir():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
    return dest

ROOT = _find_repo()
os.chdir(ROOT)
print("repo:", ROOT)

# Colab does not ship Z3; install it if this machine lacks it.
try:
    import z3
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "z3-solver"], check=True)
    import z3
print("Z3", z3.get_version_string())

HD = ROOT / "homework" / HW
PYD, CD, TD = HD / "python", HD / "c", HD / "tests"

def show(path):
    path = pathlib.Path(path)
    print(f"--- {path.name} " + "-" * max(0, 60 - len(path.name)))
    print(path.read_text().rstrip())
    print()

def run(path, *args):
    """Run one starter (from its own folder) and print its output."""
    path = pathlib.Path(path)
    r = subprocess.run([sys.executable, str(path), *map(str, args)], capture_output=True, text=True, cwd=str(path.parent))
    print(r.stdout.rstrip())
    if r.returncode != 0:
        print(r.stderr.rstrip(), file=sys.stderr)
        raise RuntimeError(f"{path} failed")
    return r.stdout

def cbmc(*args):
    """Run a CBMC tool from the c/ folder: cbmc("cbmc", "counter.c", "counter_check.c", "--unwind", 26)."""
    import importlib
    sys.path.insert(0, str(PYD))
    cli = importlib.import_module("cbmc_cli")
    out = cli.run(*args, cwd=str(CD))
    print(out.rstrip())
    return out

print("ready -- helpers: show(path), run(path, *args), cbmc(tool, *args)")

In [ ]:
# --- CBMC: the official 6.11.0 release, fetched once (about 70 MB) and checked by SHA-256.
out = run(PYD / "cbmc_cli.py")
assert "CBMC 6.11.0" in out

## Part 1 — CBMC on the counter

The harness checks `x <= 10` after each of 25 steps. Make your Part 1 edits in `c/counter_check.c`
(or copy it), and pass different `--unwind` values here.

In [ ]:
show(CD / "counter_check.c")
out = cbmc("cbmc", "counter.c", "counter_check.c", "--unwind", 26)
assert "VERIFICATION SUCCESSFUL" in out

## Part 2a — `array_max`: three conditions, in Z3

Put your candidate invariants in `CANDIDATES` in `python/hw3_part2_vcs.py`. The placeholder only shows
the output format.

In [ ]:
out = run(PYD / "hw3_part2_vcs.py")
assert "3 sufficient" in out

## Part 2b — `converge.c`: a loop contract

The argument is the invariant CBMC checks. `"1"` (true) is the placeholder.

In [ ]:
show(CD / "converge.c")
out = run(PYD / "hw3_part2_converge.py", "1")
assert "Check that loop invariant is preserved" in out

## Part 3 — signs and intervals

`absint.py` is the session 18 interpreter. P5 is yours: add it to `PROGRAMS`, and write it in C in
`c/part3_p5.c` for `goto-analyzer` and CBMC.

In [ ]:
out = run(PYD / "hw3_part3_programs.py")
assert "P4" in out

In [ ]:
show(CD / "part3_p5.c")
cbmc("goto-analyzer", "part3_p5.c", "--verify", "--intervals")
cbmc("cbmc", "part3_p5.c")

## Part 4 — generated tests, coverage, and a mutant

Generate the suite with CBMC, copy its inputs into `tests/cbmc.txt` (one `a b c` per line), put an
LLM's tests in `tests/llm.txt`, and measure both.

In [ ]:
cbmc("cbmc", "triangle.c", "--function", "triangle", "--cover", "branch", "--show-test-suite")
out = run(PYD / "hw3_part4_coverage.py", TD / "example.txt")
assert "branches" in out

## Part 5 — contracts, mutants, and vacuity

The LLM's contracts go in `c/contracts.h`. The placeholders (`1`) hold for every function, catch no
mutant, and constrain nothing.

In [ ]:
show(CD / "contracts.h")
out = run(PYD / "hw3_part5_check.py")
assert "median3" in out